# Conditional autoencoder pricing errors and significance

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


# **load data**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pandas pyarrow scikit-learn joblib tqdm tables

import pandas as pd
import numpy as np
from sklearn.preprocessing import quantile_transform, StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/stock_characteristics_monthly_py.parquet'

# Load the Parquet file into a DataFrame
df = pd.read_parquet(file_path, engine='pyarrow')

# Check the loaded data
print(df.head())

# Rename the 'month' column to 'date'
df = df.rename(columns={'month': 'date'})

# Ensure 'df' has a MultiIndex ['date', 'permno']
df = df.set_index(['date', 'permno'])

# Sort the MultiIndex
df = df.sort_index()

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
characteristics = [f'characteristic_{code}' for code in all_codes]

# Filter for dates starting from '1993'
df = df.loc[pd.IndexSlice['1975':, :], :]

# Create forward returns
df['fwd_ret_excess'] = df['ret_excess'].unstack('permno').shift(-1).stack(dropna=False)

# # Define the function to apply quantile transformation for a single date
# def process_date(date, df, characteristics):
#     group = df.loc[date, characteristics]
#     transformed = quantile_transform(group, copy=True, n_quantiles=group.shape[0])
#     transformed_df = pd.DataFrame(transformed, columns=characteristics, index=group.index)
#     transformed_df = transformed_df.mul(2).sub(1)
#     transformed_df['date'] = date  # Add the date back to the DataFrame
#     return transformed_df

# # Initialize the progress bar
# dates = df.index.get_level_values('date').unique()
# pbar = tqdm(dates, desc="Processing Dates")

# # Apply the transformation with parallel processing
# normalized_data = Parallel(n_jobs=-1)(delayed(process_date)(date, df, characteristics) for date in pbar)

# # Concatenate all the normalized data back into a single DataFrame
# normalized_df = pd.concat(normalized_data)

# # Reset and set the index to match the original DataFrame's multi-index
# normalized_df = normalized_df.reset_index().set_index(['date', 'permno'])

# # Ensure the index order matches the original DataFrame
# normalized_df = normalized_df.reindex(df.index)

# # Assign the normalized values back to the original DataFrame
# df.loc[:, characteristics] = normalized_df[characteristics].values

# # Close the progress bar
# pbar.close()

# # Scale the variables
# scaler = StandardScaler()
# df[characteristics] = scaler.fit_transform(df[characteristics])

# Clip the returns columns
df.loc[:, ['ret_excess', 'fwd_ret_excess']] = df.loc[:, ['ret_excess', 'fwd_ret_excess']].clip(lower=-1, upper=1.0)

# Print the first few rows of the scaled DataFrame to verify
print(df.head())

# Add 'ret_excess' to the list of columns
columns_to_select = characteristics

# Select these columns from the DataFrame
characteristics_data = df[columns_to_select]

# Define a function to fill missing values with the median for each group
def fill_missing_with_median(group):
    return group.fillna(group.median())

# Apply the function to each group (each date) and preserve the index
df_filled = characteristics_data.groupby(level='date').apply(fill_missing_with_median)

# Remove the additional 'date' level in the index introduced by groupby
df_filled.index = df_filled.index.droplevel(0)

# Add the filled columns back to the original DataFrame
df.update(df_filled)

print(df)

# Display the updated DataFrame to check the results
print(df.head())


In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

def load_model_predictions(results_path):
    all_predictions = {}
    file_path = results_path / 'all_predictions.h5'

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist")

    with pd.HDFStore(str(file_path), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true, xt_pred

base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3'
]
#_ret_excess_fwd
hidden_layers_list = [0, 1, 2, 3]

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    try:
        all_predictions = load_model_predictions(results_path)
    except FileNotFoundError as e:
        print(e)
        continue

    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true, xt_pred = calculate_xt_for_model(all_predictions, df, hidden_layer)
    # print(f'XT TRUE: {xt_true}')
    # print(f'XT PRED: {xt_pred}')
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


In [ ]:

############################
## CALCULATE XT FOR TPRED ##
############################

# Function to calculate x_t for a given month using Ridge Regression
def calculate_xt_y_pred(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_pred'].values
    characteristic_columns = df_month.drop(columns=['y_pred']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_pred']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_pred(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_pred(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

############################
## CALCULATE XT FOR YTRUE ##
############################

def calculate_xt_y_true(df, date, alpha=0):
    # Filter DataFrame for the given date
    df_month = df.loc[date]

    # Separate returns and characteristics
    r_t = df_month['y_true'].values
    characteristic_columns = df_month.drop(columns=['y_true']).columns
    Z_t_minus_1 = df_month.drop(columns=['y_true']).values

    # Calculate (Z_{t-1}'Z_{t-1})^{-1}Z_{t-1}r_t using Ridge Regression
    Z_t_minus_1_T = Z_t_minus_1.T
    Z_t_minus_1_cov = Z_t_minus_1_T @ Z_t_minus_1
    Z_t_minus_1_cov_reg = Z_t_minus_1_cov + alpha * np.identity(Z_t_minus_1_cov.shape[0])

    Z_t_minus_1_cov_inv = np.linalg.inv(Z_t_minus_1_cov_reg)

    # Calculate x_t
    x_t = Z_t_minus_1_cov_inv @ Z_t_minus_1_T @ r_t

    return x_t, characteristic_columns

# Calculate x_t for all unique dates in the dataset
def calculate_xt_all_dates_true(df, alpha=1e-6):
    xt_results = {}
    characteristic_columns = None

    unique_dates = df.index.get_level_values('date').unique()
    for date in unique_dates:
        x_t, characteristic_columns = calculate_xt_y_true(df, date, alpha)
        xt_results[date] = x_t

    # Convert dictionary to DataFrame
    xt_df = pd.DataFrame.from_dict(xt_results, orient='index')

    # Assign original characteristic names as column names
    if characteristic_columns is not None:
        xt_df.columns = characteristic_columns

    # Preserve the index name
    xt_df.index.name = 'date'

    return xt_df

########################
## CALCULATION OF R^2 ##
########################

def calculate_r2(xt_true, xt_pred):
    # Flatten the dataframes to create a single list of all values
    y_true = xt_true.values.flatten()
    y_pred = xt_pred.values.flatten()

    # Calculate residuals
    residuals = y_true - y_pred

    # Calculate the sum of squares of residuals
    ssr = np.sum(np.square(residuals))

    # Calculate the total sum of squares
    tss = np.sum(np.square(y_true - np.mean(y_true)))

    # Calculate R^2
    r_squared = 1 - (ssr / tss)

    return r_squared

# Define the base paths for different models
base_paths = [
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_3',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_2',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_1',
    '/content/drive/My Drive/Tidy Finance ML dataset/TPU_results_CA_0'
]

# Define the hidden layers corresponding to each base path
hidden_layers_list = [0, 1, 2, 3]

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

import pandas as pd
import numpy as np
from scipy import stats

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        # t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)
        t_critical = 3.0


        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true, xt_pred

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true, xt_pred = calculate_xt_for_model(all_predictions, df, hidden_layer)
    # print(f'XT TRUE: {xt_true}')
    # print(f'XT PRED: {xt_pred}')
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)

average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')



# **Iteration 1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[key] = expected_xt_true
        alpha_dict[key] = alpha

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], alpha=0.7)
    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


# **Iteration 2**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}
    t_alpha_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[key] = expected_xt_true
        alpha_dict[key] = alpha
        t_alpha_dict[key] = t_alpha

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict, t_alpha_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}
t_alpha_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict, t_alpha_dict = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)
    t_alpha_all_dict.update(t_alpha_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key with color grading for t_alpha
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    sc = plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], c=np.abs(t_alpha_all_dict[key]), cmap='viridis', alpha=0.7)
    plt.colorbar(sc, label='|t_alpha|')
    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


# **Iteration 3**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}
    t_alpha_dict = {}
    significant_mask_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_mask = np.abs(t_alpha) > t_critical

        significant_alpha_count = significant_alpha_mask.sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[key] = expected_xt_true
        alpha_dict[key] = alpha
        t_alpha_dict[key] = t_alpha
        significant_mask_dict[key] = significant_alpha_mask

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}
t_alpha_all_dict = {}
significant_mask_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)
    t_alpha_all_dict.update(t_alpha_dict)
    significant_mask_all_dict.update(significant_mask_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key with color grading for significant alphas
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask_all_dict[key], 'red', 'blue')
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], c=colors, alpha=0.7)
    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


In [ ]:
key

**iteration 4**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}
    t_alpha_dict = {}
    significant_mask_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_mask = np.abs(t_alpha) > t_critical

        significant_alpha_count = significant_alpha_mask.sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[key] = expected_xt_true
        alpha_dict[key] = alpha
        t_alpha_dict[key] = t_alpha
        significant_mask_dict[key] = significant_alpha_mask

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}
t_alpha_all_dict = {}
significant_mask_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)
    t_alpha_all_dict.update(t_alpha_dict)
    significant_mask_all_dict.update(significant_mask_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key with color grading for significant alphas
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask_all_dict[key], 'red', 'blue')
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], c=colors, alpha=0.7)

    # Create legend with the number of significant alphas
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask_all_dict[key].sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask_all_dict[key]) - significant_mask_all_dict[key].sum()}',
                            markerfacecolor='blue', markersize=10)
    plt.legend(handles=[red_patch, blue_patch])

    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


# **Iteration 5**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer, base_path):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}
    t_alpha_dict = {}
    significant_mask_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_mask = np.abs(t_alpha) > t_critical

        significant_alpha_count = significant_alpha_mask.sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[f'{key} (CA_{hidden_layer})'] = expected_xt_true
        alpha_dict[f'{key} (CA_{hidden_layer})'] = alpha
        t_alpha_dict[f'{key} (CA_{hidden_layer})'] = t_alpha
        significant_mask_dict[f'{key} (CA_{hidden_layer})'] = significant_alpha_mask

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}
t_alpha_all_dict = {}
significant_mask_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict = calculate_xt_for_model(all_predictions, df, hidden_layer, Path(path))
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)
    t_alpha_all_dict.update(t_alpha_dict)
    significant_mask_all_dict.update(significant_mask_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key with color grading for significant alphas
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask_all_dict[key], 'red', 'blue')
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], c=colors, alpha=0.7)

    # Create legend with the number of significant alphas
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask_all_dict[key].sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask_all_dict[key]) - significant_mask_all_dict[key].sum()}',
                            markerfacecolor='blue', markersize=10)
    plt.legend(handles=[red_patch, blue_patch])

    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


# **Iteration 6**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import os

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer, base_path):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}
    t_alpha_dict = {}
    significant_mask_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        # t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        t_critical = 3.0

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_mask = np.abs(t_alpha) > t_critical

        significant_alpha_count = significant_alpha_mask.sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[f'{key} (CA_{hidden_layer})'] = expected_xt_true
        alpha_dict[f'{key} (CA_{hidden_layer})'] = alpha
        t_alpha_dict[f'{key} (CA_{hidden_layer})'] = t_alpha
        significant_mask_dict[f'{key} (CA_{hidden_layer})'] = significant_alpha_mask

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}
t_alpha_all_dict = {}
significant_mask_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict, t_alpha_dict, significant_mask_dict = calculate_xt_for_model(all_predictions, df, hidden_layer, Path(path))
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)
    t_alpha_all_dict.update(t_alpha_dict)
    significant_mask_all_dict.update(significant_mask_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key with color grading for significant alphas and fitted line
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    colors = np.where(significant_mask_all_dict[key], 'red', 'blue')
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], c=colors, alpha=0.7)

    # Fitting a line to the data
    p = np.polyfit(xt_true_all_dict[key], alpha_all_dict[key], 1)
    plt.plot(xt_true_all_dict[key], np.polyval(p, xt_true_all_dict[key]), color='green')

    # Create legend with the number of significant alphas
    red_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Significant Alphas: {significant_mask_all_dict[key].sum()}',
                           markerfacecolor='red', markersize=10)
    blue_patch = plt.Line2D([0], [0], marker='o', color='w', label=f'Non-Significant Alphas: {len(significant_mask_all_dict[key]) - significant_mask_all_dict[key].sum()}',
                            markerfacecolor='blue', markersize=10)
    line_patch = plt.Line2D([0], [0], color='green', label='Fitted Line')
    plt.legend(handles=[red_patch, blue_patch, line_patch])

    plt.xlabel('Realized returns portfolios in $x_t$')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. realized returns for characteristics-managed portfolios in $x_t$ for {key}')
    plt.grid(True)

    # Create the results path if it doesn't exist
    os.makedirs(results_path, exist_ok=True)
    plt.savefig(str('/content/drive/My Drive/Tidy Finance ML dataset/Alpha_scatter_plots/'f'Scatter_plot_for_{key.replace("/", "_")}.png'))
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


In [ ]:
results_path

In [ ]:
results_path


In [ ]:
print(f'MODELS R^2')


# Generate the LaTeX table from the DataFrame
latex_table1 = models_df.to_latex(escape=False)

print(latex_table1)


print(f'NUMBERS OF SIGNIFICANT ALPHAS')

# Generate the LaTeX table from the DataFrame
latex_table2 = significant_alpha_df.to_latex(escape=False)

print(latex_table2)
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')

# Generate the LaTeX table from the DataFrame
latex_table3 = average_absolute_alphas_df.to_latex(escape=False)
print(latex_table3)

print(f'AVG CRITICAL T SCORE OF ALPHAS')
# Generate the LaTeX table from the DataFrame
latex_table4 = average_critical_t_score_of_alpha_df.to_latex(escape=False)

print(latex_table4)
print(f'PROPORTION OF SIGNIFICANT ALPHAS')
# Generate the LaTeX table from the DataFrame
latex_table5 = proportion_of_significant_alphas.to_latex(escape=False)

print(latex_table5)










In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    alphamale = pd.DataFrame()
    avg_t_critical_alpha = pd.DataFrame()
    prop_of_sign_ts = pd.DataFrame()
    xt_true_dict = {}
    alpha_dict = {}

    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)

        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        alpha = expected_xt_true - expected_xt_pred

        avg_abs_alpha = np.abs(alpha).mean()

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation

        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)

        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1

        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        avg_abs_t_critical = np.abs(t_alpha).mean()

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()

        share_significant_alphas = significant_alpha_count / 94
        print(f'SIGNIFICANT ALPHA SHAPE {significant_alpha_count}')

        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
        alphamale[f'{i}'] = [avg_abs_alpha]
        avg_t_critical_alpha[f'{i}'] = [avg_abs_t_critical]
        prop_of_sign_ts[f'{i}'] = [share_significant_alphas]

        xt_true_dict[key] = expected_xt_true
        alpha_dict[key] = alpha

    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']
    alphamale.index = [f'$CA_{hidden_layer}$']
    avg_t_critical_alpha.index = [f'$CA_{hidden_layer}$']
    prop_of_sign_ts.index = [f'$CA_{hidden_layer}$']
    return r2_df, significant_alpha_counts, alphamale, avg_t_critical_alpha, prop_of_sign_ts, xt_true_dict, alpha_dict

models = []
significant_alpha_counts_list = []
alphamale_list = []
avg_t_critical_alpha_list = []
prop_of_sign_ts_list = []
xt_true_all_dict = {}
alpha_all_dict = {}

for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, average_absolute_alpha, average_critical_alpha, proportion_significant_alphas, xt_true_dict, alpha_dict = calculate_xt_for_model(all_predictions, df, hidden_layer)
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
    alphamale_list.append(average_absolute_alpha)
    avg_t_critical_alpha_list.append(average_critical_alpha)
    prop_of_sign_ts_list.append(proportion_significant_alphas)
    xt_true_all_dict.update(xt_true_dict)
    alpha_all_dict.update(alpha_dict)

# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)
average_absolute_alphas_df = pd.concat(alphamale_list, axis=0)
average_critical_t_score_of_alpha_df = pd.concat(avg_t_critical_alpha_list, axis=0)
proportion_of_significant_alphas = pd.concat(prop_of_sign_ts_list, axis=0)

# Plot the scatter plot for each key
for key in xt_true_all_dict.keys():
    plt.figure(figsize=(10, 6))
    plt.scatter(xt_true_all_dict[key], alpha_all_dict[key], alpha=0.7)
    plt.xlabel('xt_true')
    plt.ylabel('Alpha')
    plt.title(f'Scatter Plot of Alphas vs. xt_true for {key}')
    plt.grid(True)
    plt.show()

print(f'MODELS XT R2{models_df}')
print(f'NUMBERS OF SIGNIFICANT ALPHAS {significant_alpha_df}')
print(f'AVG ABSOLUTE ALPHAS{average_absolute_alphas_df}')
print(f'AVG CRITICAL T SCORE OF ALPHAS{average_critical_t_score_of_alpha_df}')
print(f'PROPORTION OF SIGNIFICANT ALPHAS{proportion_of_significant_alphas}')


In [ ]:

def load_model_predictions(results_path):
    all_predictions = {}
    with pd.HDFStore(str(results_path / 'all_predictions.h5'), mode='r') as store:
        keys = store.keys()
        for key in keys:
            all_predictions[key] = store[key]
    return all_predictions

def calculate_xt_for_model(all_predictions, characteristic_df, hidden_layer):
    r2_df = pd.DataFrame()
    significant_alpha_counts = pd.DataFrame()  # Initialize an empty DataFrame
    for i, (key, df_pred) in enumerate(all_predictions.items(), start=1):
        print(key)
        df_pred = df_pred.sort_index()
        # Inner merge on MultiIndex
        merged_df = pd.merge(df_pred, characteristic_df, left_index=True, right_index=True, how='inner')

        columns_to_select = characteristics + ["y_pred"]
        characteristics_data_pred = merged_df[columns_to_select]

        columns_to_select = characteristics + ["y_true"]
        characteristics_data_true = merged_df[columns_to_select]

        # Example usage
        xt_true = calculate_xt_all_dates_true(characteristics_data_true)
        print(f'XT TRUE SHAPE {xt_true.shape}')

        xt_pred = calculate_xt_all_dates_pred(characteristics_data_pred)
        xt_r2 = calculate_r2(xt_true, xt_pred)


        expected_xt_true = xt_true.mean()
        expected_xt_pred = xt_pred.mean()
        print(f'EXPECTED XT TRUE SHAPE {expected_xt_true.shape}')

        # Calculate differences (epsilon)
        epsilon = xt_true - expected_xt_pred
        print(f'EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate mean difference (alpha)
        alpha_epsilon = epsilon.mean()
        print(f'ALPHA EPSILON XT TRUE SHAPE {epsilon.shape}')

        # Calculate standard deviation of the differences (epsilon)
        s_epsilon = epsilon.std(ddof=1)  # Sample standard deviation


        # Calculate standard error of the mean difference (alpha)
        n = len(epsilon)
        SE_alpha = s_epsilon / np.sqrt(n)


        # Calculate t-statistic for alpha
        t_alpha = alpha_epsilon / SE_alpha

        # Calculate critical value from t-distribution for 95% confidence level
        alpha_level = 0.05
        degrees_of_freedom = n - 1
        t_critical = stats.t.ppf(1 - alpha_level / 2, degrees_of_freedom)

        print(f'T-Critical: {t_critical}')
        # Determine if each t_alpha is significant
        significant_alpha_count = (np.abs(t_alpha) > t_critical).sum()
        print(f'SIGNIICANT ALPHA SHAPE {significant_alpha_count}')


        print(f"$R^2$ for $x_t$ for {key}: {xt_r2}")

        # Add the R^2 value as a new column to the DataFrame
        r2_df[f'{i}'] = [xt_r2]
        significant_alpha_counts[f'{i}'] = [significant_alpha_count]
    # Set the row name
    r2_df.index = [f'$CA_{hidden_layer}$']
    significant_alpha_counts.index = [f'$CA_{hidden_layer}$']

    return r2_df, significant_alpha_counts, xt_true, xt_pred

models = []
significant_alpha_counts_list = []
for path, hidden_layer in zip(base_paths, hidden_layers_list):
    results_path = Path(path) / 'results' / 'asset_pricing'
    all_predictions = load_model_predictions(results_path)
    r2_model, significant_alphas, xt_true, xt_pred = calculate_xt_for_model(all_predictions, df, hidden_layer)
    print(f'XT TRUE: {xt_true}')
    print(f'XT PRED: {xt_pred}')
    models.append(r2_model)
    significant_alpha_counts_list.append(significant_alphas)
# Concatenate all DataFrames in the models list by rows
models_df = pd.concat(models, axis=0)
significant_alpha_df = pd.concat(significant_alpha_counts_list, axis=0)

print(f'MODELS XT R2{models_df}')
print(f'SIGNIFICANT NUMBER OF ALPHAS{significant_alpha_df}')